# RMSX Molstar Clean Import Test

This notebook installs RMSX from the `rmsx_molstar` branch, downloads small real RMSX/Flipbook test slice PDBs from GitHub, and renders Molstar inline in Colab. It intentionally contains no embedded viewer fallback or large embedded data payload.


In [ ]:
from pathlib import Path
import json
import sys
import urllib.request

IN_COLAB = "google.colab" in sys.modules
OUTPUT_ROOT = Path("/content/rmsx_molstar_clean_import") if IN_COLAB else Path("rmsx_molstar_clean_import")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

GITHUB_REF = "rmsx_molstar"
INSTALL_FROM_GITHUB = IN_COLAB

if INSTALL_FROM_GITHUB:
    %pip install -q --upgrade "git+https://github.com/AntunesLab/rmsx.git@{GITHUB_REF}"

print("Running in Colab:", IN_COLAB)
print("Output root:", OUTPUT_ROOT.resolve())


In [ ]:
import rmsx
from rmsx import build_molstar_manifest, write_molstar_flipbook, run_flipbook

print("rmsx:", Path(rmsx.__file__).resolve())
print("Molstar helpers imported cleanly.")


## Download Real Test Fixtures

These are real RMSX/Flipbook slice PDB outputs from the repo: 1UBQ and protease combined. The downloader keeps this notebook small and makes sure the data matches the branch being tested.


In [ ]:
import shutil

RAW_BASE = f"https://raw.githubusercontent.com/AntunesLab/rmsx/{GITHUB_REF}"
FIXTURE_ROOT = OUTPUT_ROOT / "real_test_fixtures"
LOCAL_REPO_ROOT = Path.cwd()

fixture_specs = {
    "1UBQ 9-slice lDDT fixture": {
        "repo_dir": "test_files/example_uqb_lddt/chain_7_lddtmap",
        "local_dir": FIXTURE_ROOT / "example_uqb_lddt_chain_7",
        "files": [*(f"slice_{i}_first_frame.pdb" for i in range(1, 10))],
    },
    "Protease combined 9-slice RMSX": {
        "repo_dir": "rmsx_demo_outputs/protease/combined",
        "local_dir": FIXTURE_ROOT / "protease_combined",
        "files": [*(f"slice_{i}_first_frame.pdb" for i in range(1, 10))],
    },
}

def download_fixture_file(repo_dir, filename, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    local_source = LOCAL_REPO_ROOT / repo_dir / filename
    if not IN_COLAB and local_source.is_file():
        shutil.copy2(local_source, dest)
        return True
    url = f"{RAW_BASE}/{repo_dir}/{filename}"
    try:
        urllib.request.urlretrieve(url, dest)
    except Exception as exc:
        if filename == "masked_residues.csv":
            return False
        raise RuntimeError(f"Could not download {url}") from exc
    return True

fixture_dirs = {}
for label, spec in fixture_specs.items():
    local_dir = spec["local_dir"]
    for filename in spec["files"]:
        download_fixture_file(spec["repo_dir"], filename, local_dir / filename)
    fixture_dirs[label] = local_dir
    print(label, len(list(local_dir.glob("slice_*_first_frame.pdb"))), "slices", local_dir)
    print("  mask:", (local_dir / "masked_residues.csv").exists())


## Manifest Check


In [ ]:
for label, path in fixture_dirs.items():
    manifest = build_molstar_manifest(path, palette="mako")
    spacing = manifest["flipbookReference"]
    print("\n" + label)
    print(json.dumps({
        "sliceCount": len(manifest["slices"]),
        "residueCount": len(manifest["residues"]),
        "domain": manifest["domain"],
        "cameraMode": manifest["molstarRenderStyle"].get("cameraMode"),
        "spacing": {
            "default": spacing["defaultSpacingFactor"],
            "min": spacing["minimumSpacingFactor"],
            "max": spacing["maximumSpacingFactor"],
            "step": spacing["spacingStep"],
        },
        "maskedResidues": manifest["maskSummary"].get("maskedResidues"),
    }, indent=2))


## Render Real Fixtures


In [ ]:
from IPython.display import HTML, display

render_cases = [
    ("1UBQ 9-slice lDDT fixture", fixture_dirs["1UBQ 9-slice lDDT fixture"], "viridis", 620),
    ("Protease combined 9-slice RMSX", fixture_dirs["Protease combined 9-slice RMSX"], "turbo", 760),
]

for label, path, palette, height in render_cases:
    result = write_molstar_flipbook(
        path,
        palette=palette,
        camera_mode="orthographic",
        output_html=OUTPUT_ROOT / f"{label.lower().replace(' ', '_').replace('-', '_')}.html",
        output_manifest=OUTPUT_ROOT / f"{label.lower().replace(' ', '_').replace('-', '_')}.json",
        asset_mode="cdn",
        iframe_height=height,
    )
    spacing = result.manifest["flipbookReference"]["defaultSpacingFactor"]
    display(HTML(f"<h4>{label}: {palette}, orthographic, spacing={spacing}</h4>"))
    display(result)
    print(result.html_path)


## Protease Camera Comparison


In [ ]:
protease_dir = fixture_dirs["Protease combined 9-slice RMSX"]
for mode in ["orthographic", "perspective"]:
    result = write_molstar_flipbook(
        protease_dir,
        palette="mako",
        camera_mode=mode,
        output_html=OUTPUT_ROOT / f"protease_camera_{mode}.html",
        output_manifest=OUTPUT_ROOT / f"protease_camera_{mode}.json",
        asset_mode="cdn",
        iframe_height=720,
    )
    display(HTML(f"<h4>Protease camera_mode={mode}</h4>"))
    display(result)
    print(mode, result.manifest["molstarRenderStyle"]["cameraMode"])
